# SBI812 — Lecture 4
## Decision Trees and Random Forests

### Learning outcomes

By the end of this lecture, a student should be able to:

1. identify the **root node**, **decision nodes**, **branches**, and **leaf nodes** of a decision tree;
2. follow one sample from the root of a tree to its predicted class;
3. explain why a decision tree tests values such as `5.15` even when `5.15` is not present in the dataset;
4. calculate **Gini impurity** for a node;
5. calculate the weighted impurity after a proposed split;
6. explain how a decision tree chooses a feature and threshold;
7. train, display, and use a decision tree with scikit-learn; and
8. explain how a random forest develops from decision trees and implement one with scikit-learn.

---

## 1. The prediction problem

Sixteen bacterial isolates were exposed to an antibiotic. Each isolate was described using two measured features:

- **efflux_expression**: expression level of an efflux-pump gene;
- **growth_rate**: growth rate measured during the experiment.

The known outcome is called **response**. It has two possible classes:

- `sensitive`: the antibiotic inhibits the isolate;
- `resistant`: the isolate continues to grow despite the antibiotic.

The aim is to learn patterns from the 16 labelled isolates and then predict the response of a new isolate.

In [13]:
import pandas as pd

data = pd.DataFrame({
    "sample": ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8",
               "S9", "S10", "S11", "S12", "S13", "S14", "S15", "S16"],
    "efflux_expression": [2.1, 2.8, 3.2, 3.7, 4.0, 4.4, 4.9, 5.4,
                          5.9, 6.4, 6.9, 6.2, 7.2, 7.9, 8.4, 9.0],
    "growth_rate": [0.72, 0.68, 0.81, 0.64, 0.76, 0.58, 0.70, 0.55,
                    0.48, 0.59, 0.52, 0.71, 0.78, 0.69, 0.75, 0.83],
    "response": ["sensitive", "sensitive", "sensitive", "sensitive",
                 "sensitive", "sensitive", "sensitive", "resistant",
                 "resistant", "resistant", "resistant", "sensitive",
                 "sensitive", "resistant", "resistant", "resistant"]
})

data

,sample,efflux_expression,growth_rate,response
0,S1,2.1,0.72,sensitive
1,S2,2.8,0.68,sensitive
2,S3,3.2,0.81,sensitive
3,S4,3.7,0.64,sensitive
4,S5,4.0,0.76,sensitive
5,S6,4.4,0.58,sensitive
6,S7,4.9,0.70,sensitive
7,S8,5.4,0.55,resistant
8,S9,5.9,0.48,resistant
9,S10,6.4,0.59,resistant


### Reading the table

Each **row** represents one isolate. Each **feature column** contains a measurement that may help with prediction. The `response` column contains the answer that the model must learn to predict; it is therefore the **target** or **class label**.

For example, S1 has an efflux-expression value of 2.1, a growth rate of 0.72, and is known to be sensitive.

Before fitting any model, it helps to see how the two classes are distributed.

In [ ]:
data["response"].value_counts()

There are 9 sensitive isolates and 7 resistant isolates. A useful model should separate these classes using the measured features.

---

## 2. What is a decision tree?

A decision tree predicts by asking a sequence of questions. Each question has two possible answers: **yes** or **no**.

Consider the following fitted tree:

```text
                              ROOT NODE
                   Is efflux_expression ≤ 5.15?
                         /                 \
                       Yes                 No
                       /                    \
              LEAF: sensitive       DECISION NODE
                                  Is growth_rate ≤ 0.70?
                                      /           \
                                    Yes            No
                                    /               \
                           LEAF: resistant     DECISION NODE
                                             Is efflux_expression ≤ 7.80?
                                                  /             \
                                                Yes              No
                                                /                 \
                                       LEAF: sensitive     LEAF: resistant
```

### The parts of the tree

| Part | Meaning in this tree |
|---|---|
| **Root node** | The first question: `efflux_expression ≤ 5.15` |
| **Decision node** | A later question, such as `growth_rate ≤ 0.70` |
| **Branch** | A path labelled Yes or No |
| **Leaf node** | The final prediction; no further question is asked |
| **Depth** | The number of decisions between the root and a node |

The symbol `≤` means **less than or equal to**. Therefore:

- if the condition is true, follow the **Yes/left** branch;
- if the condition is false, follow the **No/right** branch.

### Following S3 through the tree

S3 has `efflux_expression = 3.2`.

1. Ask: is `3.2 ≤ 5.15`?
2. The answer is **Yes**.
3. Follow the left branch.
4. The leaf predicts **sensitive**.

The tree does not need to examine S3's growth rate because it has already reached a leaf.

### Following S15 through the tree

S15 has `efflux_expression = 8.4` and `growth_rate = 0.75`.

1. Is `8.4 ≤ 5.15`? **No** → move right.
2. Is `0.75 ≤ 0.70`? **No** → move right.
3. Is `8.4 ≤ 7.80`? **No** → move right.
4. The final prediction is **resistant**.

The model's prediction is determined by the complete path, not by one feature considered in isolation.

---

## 3. Where do thresholds such as 5.15 come from?

The value `5.15` was not measured for any isolate. It is a **midpoint** placed between two neighbouring measured values.

The procedure begins by sorting one feature from the smallest value to the largest. The efflux-expression values in the dataset are:

```text
2.1, 2.8, 3.2, 3.7, 4.0, 4.4, 4.9, 5.4,
5.9, 6.2, 6.4, 6.9, 7.2, 7.9, 8.4, 9.0
```

Notice that `6.2` must appear before `6.4` and `6.9`. The original row order does not matter; threshold search uses the **sorted feature values**.

The feature values become more informative when they are displayed together with their known response labels:

| Sorted position | Efflux expression | Known response |
|---:|---:|---|
| 1 | 2.1 | sensitive |
| 2 | 2.8 | sensitive |
| 3 | 3.2 | sensitive |
| 4 | 3.7 | sensitive |
| 5 | 4.0 | sensitive |
| 6 | 4.4 | sensitive |
| 7 | **4.9** | **sensitive** |
| 8 | **5.4** | **resistant** |
| 9 | 5.9 | resistant |
| 10 | 6.2 | sensitive |
| 11 | 6.4 | resistant |
| 12 | 6.9 | resistant |
| 13 | 7.2 | sensitive |
| 14 | 7.9 | resistant |
| 15 | 8.4 | resistant |
| 16 | 9.0 | resistant |

The values `4.9` and `5.4` were not selected arbitrarily:

- they are adjacent after sorting;
- `4.9` is the final value in the first continuous group of sensitive isolates;
- the next observed value, `5.4`, belongs to a resistant isolate; and
- placing a boundary between them sends the first seven sensitive isolates to the same child node.

The boundary can be pictured as:

```text
2.1  2.8  3.2  3.7  4.0  4.4  4.9  |  5.4  5.9  6.2 ...
 S    S    S    S    S    S    S    |   R    R    S  ...
                                      ↑
                           possible dividing position
```

Here, `S` means sensitive and `R` means resistant. A threshold must be placed between measured values rather than directly on one of the observations. The midpoint between `4.9` and `5.4` is therefore tested.

For each pair of neighbouring values, calculate:

$$
\text{midpoint}=\frac{\text{left value}+\text{right value}}{2}
$$

For example, `4.9` and `5.4` are neighbours:

$$
\frac{4.9+5.4}{2}=\frac{10.3}{2}=5.15
$$

Therefore, the question `efflux_expression ≤ 5.15` places all observations up to 4.9 on the left and all observations from 5.4 upward on the right.

### All candidate midpoints

| Neighbouring values | Calculation | Candidate threshold |
|---:|:---:|---:|
| 2.1 and 2.8 | (2.1 + 2.8) / 2 | 2.45 |
| 2.8 and 3.2 | (2.8 + 3.2) / 2 | 3.00 |
| 3.2 and 3.7 | (3.2 + 3.7) / 2 | 3.45 |
| 3.7 and 4.0 | (3.7 + 4.0) / 2 | 3.85 |
| 4.0 and 4.4 | (4.0 + 4.4) / 2 | 4.20 |
| 4.4 and 4.9 | (4.4 + 4.9) / 2 | 4.65 |
| 4.9 and 5.4 | (4.9 + 5.4) / 2 | **5.15** |
| 5.4 and 5.9 | (5.4 + 5.9) / 2 | 5.65 |
| 5.9 and 6.2 | (5.9 + 6.2) / 2 | 6.05 |
| 6.2 and 6.4 | (6.2 + 6.4) / 2 | 6.30 |
| 6.4 and 6.9 | (6.4 + 6.9) / 2 | **6.65** |
| 6.9 and 7.2 | (6.9 + 7.2) / 2 | 7.05 |
| 7.2 and 7.9 | (7.2 + 7.9) / 2 | **7.55** |
| 7.9 and 8.4 | (7.9 + 8.4) / 2 | 8.15 |
| 8.4 and 9.0 | (8.4 + 9.0) / 2 | 8.70 |

These are **candidate** thresholds. Calculating a midpoint does not mean that the tree will use it. The tree evaluates the separation produced by every candidate and selects the best one.

In [ ]:
# Generate the candidate thresholds rather than calculating them by hand.
sorted_values = sorted(data["efflux_expression"].unique())

midpoints = []

for position in range(len(sorted_values) - 1):
    left_value = sorted_values[position]
    right_value = sorted_values[position + 1]
    midpoint = (left_value + right_value) / 2
    midpoints.append(midpoint)

midpoints

### Understanding the midpoint code

- `unique()` keeps one copy of each distinct value.
- `sorted(...)` arranges the values from smallest to largest.
- `len(sorted_values) - 1` is used because the last value has no value after it.
- `position` is an index: 0, 1, 2, and so on.
- `sorted_values[position]` obtains the value at the current position.
- `sorted_values[position + 1]` obtains its next neighbour.
- The two values are averaged and the result is appended to `midpoints`.

The class labels have not yet been used. The feature values generate the possible thresholds; the class labels are then used to judge which threshold separates the classes best.

The algorithm does not immediately choose `4.9` and `5.4` simply because their labels differ. It generates all valid candidate midpoints and calculates the quality of every resulting split. The boundary between `4.9` and `5.4` wins at the root because its midpoint, `5.15`, gives the largest decrease in weighted Gini impurity.

---

## 4. Measuring the mixture in a node: Gini impurity

A node is **pure** when every sample in it belongs to the same class. A node is **impure** when it contains a mixture of classes.

Gini impurity measures this mixture:

$$
Gini = 1 - \sum p_i^2
$$

Here, $p_i$ is the proportion belonging to class $i$.

With two classes, sensitive and resistant:

$$
Gini = 1 - p_{sensitive}^2 - p_{resistant}^2
$$

### Example A: a pure node

Suppose a node contains 7 sensitive isolates and 0 resistant isolates.

$$
p_{sensitive}=\frac{7}{7}=1
$$

$$
p_{resistant}=\frac{0}{7}=0
$$

$$
Gini=1-(1)^2-(0)^2=0
$$

A Gini impurity of **0** means that the node is completely pure.

### Example B: a mixed node

Suppose a node contains 2 sensitive and 7 resistant isolates. There are 9 isolates in total.

$$
p_{sensitive}=\frac{2}{9}, \qquad p_{resistant}=\frac{7}{9}
$$

$$
Gini=1-\left(\frac{2}{9}\right)^2-\left(\frac{7}{9}\right)^2
$$

$$
Gini=1-\frac{4}{81}-\frac{49}{81}=\frac{28}{81}=0.3457
$$

The value is greater than zero because both classes are present.

### Interpreting Gini values for two classes

| Gini value | Interpretation |
|---:|---|
| 0 | Only one class is present |
| Near 0 | One class strongly dominates |
| 0.5 | The two classes are evenly mixed |

A decision tree looks for a split that produces child nodes with lower impurity than the parent node.

In [ ]:
def gini_impurity(sensitive_count, resistant_count):
    total = sensitive_count + resistant_count

    if total == 0:
        return 0

    sensitive_proportion = sensitive_count / total
    resistant_proportion = resistant_count / total

    gini = 1 - sensitive_proportion**2 - resistant_proportion**2
    return gini

print("Pure node:", gini_impurity(7, 0))
print("Mixed node:", round(gini_impurity(2, 7), 4))

The function receives the number of samples in each class. It converts the counts into proportions, squares the proportions, and applies the Gini formula. The `if total == 0` condition prevents division by zero if an empty group is supplied.

---

## 5. Evaluating the root before splitting

At the root, all 16 isolates are together:

- sensitive = 9;
- resistant = 7;
- total = 16.

Therefore:

$$
Gini_{root}=1-\left(\frac{9}{16}\right)^2-\left(\frac{7}{16}\right)^2
$$

$$
Gini_{root}=1-0.3164-0.1914=0.4922
$$

This value is close to 0.5 because the root contains substantial numbers of both classes.

## 6. Testing the candidate threshold 5.15

The rule is:

```text
efflux_expression ≤ 5.15
```

The split produces:

| Child node | Samples | Sensitive | Resistant | Gini |
|---|---|---:|---:|---:|
| Left: value ≤ 5.15 | S1–S7 | 7 | 0 | 0.0000 |
| Right: value > 5.15 | S8–S16 | 2 | 7 | 0.3457 |

The two child nodes are not the same size. Their impurities must therefore be combined using a **weighted average**.

$$
Gini_{split}=\frac{n_{left}}{n_{total}}Gini_{left}
             +\frac{n_{right}}{n_{total}}Gini_{right}
$$

For this split:

$$
Gini_{split}=\frac{7}{16}(0)+\frac{9}{16}(0.3457)=0.1944
$$

The decrease in impurity is:

$$
\text{impurity decrease}=Gini_{parent}-Gini_{split}
$$

$$
=0.4922-0.1944=0.2978
$$

The split has moved seven sensitive isolates into a completely pure left node. This explains its large reduction in impurity.

## 7. Comparing 5.15 with another candidate: 6.65

The threshold 6.65 is the midpoint of 6.4 and 6.9:

$$
\frac{6.4+6.9}{2}=6.65
$$

Using `efflux_expression ≤ 6.65` produces:

| Child node | Sensitive | Resistant | Total | Gini |
|---|---:|---:|---:|---:|
| Left | 8 | 3 | 11 | 0.3967 |
| Right | 1 | 4 | 5 | 0.3200 |

Weighted impurity:

$$
Gini_{split}=\frac{11}{16}(0.3967)+\frac{5}{16}(0.3200)=0.3727
$$

Impurity decrease:

$$
0.4922-0.3727=0.1195
$$

### Which threshold is better?

| Threshold | Weighted impurity after split | Impurity decrease |
|---:|---:|---:|
| **5.15** | **0.1944** | **0.2978** |
| 6.65 | 0.3727 | 0.1195 |

The threshold **5.15** is better because:

- its weighted impurity is lower; and
- its impurity decrease is larger.

Software repeats this comparison for all candidate thresholds of all available features. The feature and threshold giving the best improvement become the decision at that node.

In [ ]:
# Examine one proposed split directly from the table.
threshold = 5.15

left_group = data[data["efflux_expression"] <= threshold]
right_group = data[data["efflux_expression"] > threshold]

print("LEFT GROUP")
print(left_group[["sample", "efflux_expression", "response"]].to_string(index=False))

print("\nRIGHT GROUP")
print(right_group[["sample", "efflux_expression", "response"]].to_string(index=False))

The expressions inside square brackets are Boolean conditions. For each row, pandas checks whether the condition is `True` or `False`. Only rows for which it is `True` are retained.

---

## 8. How the remaining branches are selected

After the first split, the left node is already pure, so it becomes a sensitive leaf. The right node still contains 2 sensitive and 7 resistant isolates, so the model searches for another split using only those nine samples.

### Second decision: `growth_rate ≤ 0.70`

The parent node contains 2 sensitive and 7 resistant isolates:

$$
Gini_{parent}=0.3457
$$

Only the nine isolates that passed through the No branch of the root are considered here. Their growth rates are sorted together with their labels:

```text
0.48 R, 0.52 R, 0.55 R, 0.59 R, 0.69 R,
0.71 S, 0.75 R, 0.78 S, 0.83 R
```

The boundary between `0.69` and `0.71` is useful because all five observations before it are resistant. Its midpoint is:

$$
\frac{0.69+0.71}{2}=0.70
$$

This gives the threshold `0.70`. The split produces:

| Child node | Sensitive | Resistant | Gini |
|---|---:|---:|---:|
| growth_rate ≤ 0.70 | 0 | 5 | 0.0000 |
| growth_rate > 0.70 | 2 | 2 | 0.5000 |

Weighted impurity:

$$
\frac{5}{9}(0)+\frac{4}{9}(0.50)=0.2222
$$

Impurity decrease:

$$
0.3457-0.2222=0.1235
$$

The left child is pure resistant. The right child contains two sensitive and two resistant isolates and therefore requires another decision.

### Third decision: `efflux_expression ≤ 7.80`

Only four isolates reach this node because their growth rate is greater than 0.70:

| Sample | Efflux expression | Growth rate | Response |
|---|---:|---:|---|
| S12 | 6.2 | 0.71 | sensitive |
| S13 | 7.2 | 0.78 | sensitive |
| S15 | 8.4 | 0.75 | resistant |
| S16 | 9.0 | 0.83 | resistant |

S14, whose efflux-expression value is `7.9`, is not in this node. Its growth rate is `0.69`, so it followed the Yes branch at the preceding decision and already entered the resistant leaf.

The sorted efflux-expression values that remain are therefore:

```text
6.2 S, 7.2 S, 8.4 R, 9.0 R
```

At this node, `7.2` and `8.4` are neighbours. Their midpoint is:

$$
\frac{7.2+8.4}{2}=7.80
$$

The four samples are divided into:

- S12 and S13: both sensitive;
- S15 and S16: both resistant.

Both children are pure, so their weighted impurity is 0. The tree has separated all training samples correctly.

---

## 9. Predicting new isolates by hand

Before using scikit-learn to make predictions, use the decision tree itself. Start at the root for each isolate and record the answer to every question encountered.

Use this tree:

```text
Is efflux_expression ≤ 5.15?
├── Yes → predict sensitive
└── No  → Is growth_rate ≤ 0.70?
           ├── Yes → predict resistant
           └── No  → Is efflux_expression ≤ 7.80?
                      ├── Yes → predict sensitive
                      └── No  → predict resistant
```

### Prediction worksheet

Complete the blank columns before running the scikit-learn prediction cells later in the notebook.

| Isolate | Efflux expression | Growth rate | Question 1: `efflux ≤ 5.15` | Question 2: `growth ≤ 0.70` | Question 3: `efflux ≤ 7.80` | Prediction |
|---|---:|---:|---|---|---|---|
| A | 4.6 | 0.80 |  | Not reached | Not reached |  |
| B | 5.8 | 0.60 |  |  | Not reached |  |
| C | 6.8 | 0.72 |  |  |  |  |
| D | 8.1 | 0.74 |  |  |  |  |
| E | 5.15 | 0.90 |  | Not reached | Not reached |  |
| F | 7.6 | 0.80 |  |  |  |  |

### Worked example using isolate B

Isolate B has efflux expression 5.8 and growth rate 0.60.

1. Is `5.8 ≤ 5.15`? **No**, so move to the right.
2. Is `0.60 ≤ 0.70`? **Yes**, so move to the left leaf.
3. The prediction is **resistant**.

Question 3 is not reached for isolate B. A tree asks only the questions encountered along that isolate's path.

### Pay attention to equality

The symbol `≤` includes equality. Therefore:

- an efflux-expression value of exactly `5.15` follows the Yes branch at the root;
- an efflux-expression value of exactly `7.80` follows the Yes branch at the third decision.

Keep the completed worksheet. It will be compared with the predictions returned by scikit-learn.

---

## 10. Implementing the decision tree with scikit-learn

Scikit-learn expects the predictors and target to be stored separately:

- `X` contains the input features;
- `y` contains the known class labels.

The capital letter `X` commonly represents a table containing several feature columns. The lowercase `y` commonly represents one target column.

In [ ]:
X = data[["efflux_expression", "growth_rate"]]
y = data["response"]

print("Features:")
display(X.head())

print("Target:")
display(y.head())

Double square brackets are used for `X` because a list of column names is being selected. The result remains a two-dimensional DataFrame. Single square brackets select the one target column as a Series.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

tree_model.fit(X, y)

### Meaning of each statement

- `from sklearn.tree import DecisionTreeClassifier` makes the classifier available in the notebook.
- `criterion="gini"` tells the model to evaluate splits using Gini impurity.
- `max_depth=3` allows at most three levels of decisions. Limiting depth can prevent an unnecessarily complicated tree.
- `random_state=42` makes any random choices reproducible. The number 42 has no biological meaning; another fixed integer could be used.
- `tree_model.fit(X, y)` trains the model. During fitting, the algorithm examines candidate features and thresholds and builds the tree.

The word **fit** means learning from the supplied examples. It does not mean merely drawing a diagram.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(15, 8))

plot_tree(
    tree_model,
    feature_names=X.columns,
    class_names=tree_model.classes_,
    filled=True,
    rounded=True,
    impurity=True,
    proportion=False
)

plt.title("Decision tree for antibiotic response")
plt.show()

### Reading a scikit-learn tree box

A displayed node normally contains:

- the decision rule, for example `efflux_expression <= 5.15`;
- `gini`, the impurity of samples reaching that node;
- `samples`, the number of training observations in that node;
- `value`, the counts belonging to the classes;
- `class`, the majority class and therefore the node's prediction.

The order of counts in `value` follows `tree_model.classes_`. Always inspect the class order rather than assuming it.

In [ ]:
print(tree_model.classes_)

## 11. Comparing hand predictions with scikit-learn

New data must contain the same feature columns used during training. Each row represents one new isolate. The following table contains the same six isolates used in the hand-prediction worksheet.

In [ ]:
new_isolates = pd.DataFrame({
    "efflux_expression": [4.6, 5.8, 6.8, 8.1, 5.15, 7.6],
    "growth_rate": [0.80, 0.60, 0.72, 0.74, 0.90, 0.80]
}, index=["A", "B", "C", "D", "E", "F"])

predicted_response = tree_model.predict(new_isolates)

prediction_table = new_isolates.copy()
prediction_table["predicted_response"] = predicted_response
prediction_table

- Compare the `predicted_response` column with the predictions written in the worksheet.
- If an answer is different, return to the root and follow that isolate's path again.
- `predict(new_isolates)` sends each new row through the fitted tree.
- The returned predictions are stored in `predicted_response`.
- `copy()` makes a separate table so that the original `new_isolates` table is not changed.
- A new column is added to show each predicted class beside its feature values.

A prediction is an output from a learned pattern. It is not proof that the isolate truly has that response; experimental validation remains important.

---

## 12. Why one decision tree may be unreliable

A decision tree is highly interpretable, but it can change when the training data change slightly. For example, replacing a few isolates may cause the algorithm to choose a different first threshold and build different later branches.

A very deep tree can also memorize details of the training data. This is called **overfitting**: performance is excellent on the examples already seen, but poorer on new samples.

Two common controls are:

- `max_depth`: limits the number of decision levels;
- `min_samples_leaf`: requires a leaf to contain at least a specified number of samples.

Another approach is to combine many trees. This leads to a random forest.

---

## 13. From one decision tree to a random forest

A **random forest** is a collection of decision trees. The trees are deliberately made different from one another.

Each tree is usually trained using:

1. a random sample of the training rows, selected with replacement; and
2. a random subset of features considered at each split.

For classification, each tree votes for a class. The forest predicts the class receiving the most votes.

```text
New isolate
   ├── Tree 1 predicts resistant
   ├── Tree 2 predicts sensitive
   ├── Tree 3 predicts resistant
   ├── Tree 4 predicts resistant
   └── Tree 5 predicts sensitive

Final forest prediction: resistant (3 votes against 2)
```

The individual trees may make different errors. Combining their votes often gives a more stable prediction than relying on one tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=3,
    random_state=42
)

forest_model.fit(X, y)

### Meaning of the random-forest code

- `RandomForestClassifier` creates a classification model made from many trees.
- `n_estimators=200` requests 200 decision trees. An estimator here means one tree.
- `max_depth=3` limits the depth of each tree.
- `random_state=42` allows the same random forest to be reproduced.
- `fit(X, y)` trains all the trees and stores them inside `forest_model`.

The hand calculation for one decision tree remains important: the trees inside a random forest use the same basic ideas of candidate thresholds and impurity reduction.

In [ ]:
forest_predictions = forest_model.predict(new_isolates)

forest_prediction_table = new_isolates.copy()
forest_prediction_table["predicted_response"] = forest_predictions
forest_prediction_table

## 14. Feature importance

A fitted random forest can report how much each feature contributed to reducing impurity across its trees. Larger values indicate greater use in the forest's decisions.

Feature importance does **not** prove that a feature causes resistance. It describes the feature's predictive contribution within this model and dataset.

In [ ]:
importance_table = pd.DataFrame({
    "feature": X.columns,
    "importance": forest_model.feature_importances_
})

importance_table = importance_table.sort_values(
    "importance",
    ascending=False
)

importance_table

---

## 15. Decision tree and random forest compared

| Property | Decision tree | Random forest |
|---|---|---|
| Number of trees | One | Many |
| Final prediction | Leaf reached in the tree | Majority vote across trees |
| Interpretation | A complete path can be followed | The complete forest is harder to inspect |
| Sensitivity to small data changes | Often higher | Usually lower |
| Risk of overfitting | High for an unrestricted deep tree | Reduced by combining varied trees |
| Feature importance | Available | Available and averaged across trees |

The choice depends on the aim. A small tree is valuable when transparent rules are required. A random forest is often preferred when predictive stability is more important than tracing one short path.

---

## 16. Summary of the complete process

1. Separate feature columns (`X`) from the target (`y`).
2. At the current node, sort the observed values of each feature.
3. Calculate midpoints between neighbouring distinct values.
4. Treat each midpoint as a candidate threshold.
5. Divide samples into left and right groups for each candidate.
6. Calculate Gini impurity for both child groups.
7. Calculate their weighted impurity.
8. Select the feature and threshold producing the greatest impurity decrease.
9. Repeat the process within impure child nodes.
10. Stop when a stopping condition is met, such as a pure node or the maximum depth.

The central idea is:

> A decision tree searches for questions that create child groups that are purer than the group before the question.

---

## 17. Practice exercises

### Exercise 1: following a tree

Use the tree shown in Section 2.

For each isolate below, write every question asked, the Yes/No answer, and the final prediction.

1. `efflux_expression = 4.6`, `growth_rate = 0.80`
2. `efflux_expression = 6.1`, `growth_rate = 0.60`
3. `efflux_expression = 7.0`, `growth_rate = 0.72`
4. `efflux_expression = 8.1`, `growth_rate = 0.70`

### Exercise 2: candidate midpoint

The neighbouring sorted values are 10.2 and 11.0.

1. Calculate their midpoint.
2. Write the decision rule using `≤`.
3. State which branch receives a sample with value 10.7.

### Exercise 3: Gini impurity

A node contains 6 sensitive and 2 resistant isolates.

1. Calculate the proportion of each class.
2. Calculate the Gini impurity.
3. State whether the node is pure or mixed.

### Exercise 4: comparing two splits

Two candidate splits produce the following weighted impurities:

- Split A: 0.18
- Split B: 0.31

The parent impurity is 0.46.

1. Calculate the impurity decrease for each split.
2. Which split should be selected?
3. Explain the reason without referring only to the letters A and B.

### Exercise 5: reading code

Explain the purpose of each item in the following code:

```python
model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)
model.fit(X, y)
```

### Exercise 6: implementation

Using the lecture dataset:

1. create a decision tree with `max_depth=2`;
2. fit it using `X` and `y`;
3. display the tree;
4. predict the response of an isolate with efflux expression 7.6 and growth rate 0.73; and
5. compare the displayed tree with the tree produced when `max_depth=3`.

### Exercise 7: random forest

Create a random forest containing 100 trees. Fit it to `X` and `y`, predict the three samples in `new_isolates`, and display the feature-importance values.